In [1]:
import numpy as np
from pathlib import Path

import torch
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
)


# =============================================================================
# Paths
# =============================================================================

BRAIN_AREA_PATH = "/home/maria/Science/data/brain_area.npy"
LABELS_PATH = "/home/maria/Science/data/image_labels.npy"
NEURAL_PATH = "/home/maria/Science/data/hybrid_neural_responses_reduced.npy"

OUTDIR = Path("/home/maria/Science/results/lm_vs_al_adam_decoder")
OUTDIR.mkdir(parents=True, exist_ok=True)

OUT_NPZ = OUTDIR / "lm_vs_al_adam_loo_results.npz"


# =============================================================================
# Settings copied from your Adam decoder script
# =============================================================================

LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 3000

RANDOM_SEED = 0
EPS = 1e-12

# Area aliases: adapt if your brain_area.npy uses different strings.
LM_ALIASES = {"LM", "VISl", "VISL", "visl", "VIS-l", "VIS_l"}
AL_ALIASES = {"AL", "VISal", "VISAL", "visal", "VIS-al", "VIS_al"}


# =============================================================================
# Utility
# =============================================================================

def sigmoid_np(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))


def load_labels(path: str) -> np.ndarray:
    obj = np.load(path, allow_pickle=True)

    # Handles either labels.npy as a plain array,
    # or image_labels.npy-style dict with key "labels".
    if isinstance(obj, np.ndarray) and obj.shape == () and isinstance(obj.item(), dict):
        d = obj.item()
        if "labels" not in d:
            raise ValueError(f"Label dict at {path} does not contain key 'labels'.")
        labels = d["labels"]
    else:
        labels = obj

    labels = np.asarray(labels).astype(np.int64).ravel()

    print(f"Loaded labels: {labels.shape}")
    print("Raw label counts including possible -1:")
    unique, counts = np.unique(labels, return_counts=True)
    print(dict(zip(unique.tolist(), counts.tolist())))

    print("Convention assumed: 0 = inanimate, 1 = animate, -1 = unlabeled/exclude")
    return labels


def load_brain_areas(path: str) -> np.ndarray:
    areas = np.load(path, allow_pickle=True)
    areas = np.asarray(areas).astype(str).ravel()

    print(f"Loaded brain areas: {areas.shape}")
    unique, counts = np.unique(areas, return_counts=True)

    print("Brain area counts:")
    for a, c in zip(unique, counts):
        print(f"  {a}: {c}")

    return areas


def load_and_align_data():
    X = np.load(NEURAL_PATH, allow_pickle=True)
    X = np.asarray(X, dtype=np.float64)

    y = load_labels(LABELS_PATH)
    brain_area = load_brain_areas(BRAIN_AREA_PATH)

    print()
    print("=" * 80)
    print("Raw data shapes")
    print("=" * 80)
    print(f"X shape:          {X.shape}")
    print(f"labels shape:     {y.shape}")
    print(f"brain_area shape: {brain_area.shape}")

    # Make X images x neurons/features.
    if X.shape[0] == len(y):
        print("[INFO] X already looks like images x features.")
    elif X.shape[1] == len(y):
        print("[INFO] Transposing X to images x features.")
        X = X.T
    else:
        raise ValueError(
            f"Cannot align X with labels. X={X.shape}, labels={y.shape}. "
            "Expected one X axis to match number of labels."
        )

    if X.shape[1] != len(brain_area):
        raise ValueError(
            f"Feature axis of X does not match brain_area length. "
            f"X features={X.shape[1]}, brain_area={len(brain_area)}. "
            "This usually means brain_area.npy belongs to a different neural matrix."
        )

    # Exclude unlabeled images.
    labeled_mask = y != -1
    original_indices = np.where(labeled_mask)[0]

    X = X[labeled_mask]
    y = y[labeled_mask]

    print()
    print("=" * 80)
    print("After excluding label -1")
    print("=" * 80)
    print(f"X labeled shape: {X.shape}")
    print(f"y labeled shape: {y.shape}")
    print(f"Label counts [inanimate, animate]: {np.bincount(y, minlength=2)}")

    # Globally remove invalid/nonconstant features, while keeping brain_area aligned.
    finite_cols = np.all(np.isfinite(X), axis=0)
    std_cols = np.std(X[:, finite_cols], axis=0) > 1e-12

    good_cols = np.zeros(X.shape[1], dtype=bool)
    finite_indices = np.where(finite_cols)[0]
    good_cols[finite_indices[std_cols]] = True

    X_clean = X[:, good_cols]
    brain_area_clean = brain_area[good_cols]

    print()
    print("=" * 80)
    print("After feature cleaning")
    print("=" * 80)
    print(f"X clean shape: {X_clean.shape}")
    print(f"Removed bad/nonconstant features: {np.sum(~good_cols)}")

    return X_clean, y, brain_area_clean, labeled_mask, original_indices, good_cols


# =============================================================================
# Adam logistic regression
# =============================================================================

class TorchLogisticRegression(torch.nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.linear = torch.nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)


def fit_adam_logistic(
    X_train,
    y_train,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    epochs=EPOCHS,
    seed=0,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    X_t = torch.tensor(X_train.astype(np.float32))
    y_t = torch.tensor(y_train.astype(np.float32)).view(-1, 1)

    model = TorchLogisticRegression(X_train.shape[1])

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    loss_fn = torch.nn.BCEWithLogitsLoss()

    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model(X_t)
        loss = loss_fn(logits, y_t)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        w = model.linear.weight.detach().cpu().numpy().ravel().astype(np.float64)
        b = float(model.linear.bias.detach().cpu().numpy()[0])

    return w, b


# =============================================================================
# LOO decoder
# =============================================================================

def run_loo_decoder_for_area(X_area, y, area_name: str):
    n, d = X_area.shape

    if d == 0:
        raise ValueError(f"No features found for area {area_name}.")

    logits = np.zeros(n, dtype=np.float64)
    probs = np.zeros(n, dtype=np.float64)
    preds = np.zeros(n, dtype=np.int64)

    print()
    print("#" * 80)
    print(f"Running LOO Adam decoder for {area_name}")
    print("#" * 80)
    print(f"X_area shape: {X_area.shape}")
    print(f"Label counts [inanimate, animate]: {np.bincount(y, minlength=2)}")

    for test_idx in range(n):
        train_mask = np.arange(n) != test_idx

        X_train_raw = X_area[train_mask]
        y_train = y[train_mask]
        X_test_raw = X_area[~train_mask]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_raw)
        X_test = scaler.transform(X_test_raw)

        w, b = fit_adam_logistic(
            X_train,
            y_train,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            epochs=EPOCHS,
            seed=10_000 + test_idx,
        )

        logit = float(X_test[0] @ w + b)
        prob = float(sigmoid_np(logit))
        pred = int(prob >= 0.5)

        logits[test_idx] = logit
        probs[test_idx] = prob
        preds[test_idx] = pred

        print(
            f"[{area_name} LOO {test_idx + 1:03d}/{n}] "
            f"true={y[test_idx]} "
            f"logit={logit:+.6f} "
            f"prob={prob:.4f} "
            f"pred={pred}"
        )

    acc = accuracy_score(y, preds)
    bal_acc = balanced_accuracy_score(y, preds)
    auc = roc_auc_score(y, probs)
    cm = confusion_matrix(y, preds, labels=[0, 1])

    print()
    print("=" * 80)
    print(f"{area_name} LOO summary")
    print("=" * 80)
    print(f"Features:          {d}")
    print(f"Accuracy:          {acc:.4f}")
    print(f"Balanced accuracy: {bal_acc:.4f}")
    print(f"AUC:               {auc:.4f}")
    print("Confusion matrix rows=true [inanimate, animate], cols=pred [inanimate, animate]:")
    print(cm)

    return {
        "area_name": area_name,
        "n_features": d,
        "logits": logits,
        "probs": probs,
        "preds": preds,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "auc": auc,
        "confusion_matrix": cm,
    }


# =============================================================================
# Paired comparison: LM vs AL
# =============================================================================

def paired_sign_flip_test(correct_lm, correct_al, n_permutations=100_000, seed=RANDOM_SEED):
    """
    Paired randomization test on per-image correctness difference.

    d_i = correct_LM_i - correct_AL_i.
    Under null, the sign of each nonzero paired difference is exchangeable.

    Tests whether LM > AL.
    """
    rng = np.random.default_rng(seed)

    d = correct_lm.astype(np.int64) - correct_al.astype(np.int64)
    observed = float(d.mean())

    nonzero = d[d != 0]

    if len(nonzero) == 0:
        return observed, np.array([0.0]), 1.0, 1.0

    null = np.zeros(n_permutations, dtype=np.float64)

    for b in range(n_permutations):
        signs = rng.choice([-1, 1], size=len(nonzero))
        d_perm = nonzero * signs
        null[b] = d_perm.sum() / len(d)

    p_one_sided_lm_gt_al = (1.0 + np.sum(null >= observed)) / (n_permutations + 1.0)
    p_two_sided = (1.0 + np.sum(np.abs(null) >= abs(observed))) / (n_permutations + 1.0)

    return observed, null, p_one_sided_lm_gt_al, p_two_sided


def summarize_comparison(y, lm, al):
    correct_lm = lm["preds"] == y
    correct_al = al["preds"] == y

    lm_only = int(np.sum(correct_lm & ~correct_al))
    al_only = int(np.sum(~correct_lm & correct_al))
    both_correct = int(np.sum(correct_lm & correct_al))
    both_wrong = int(np.sum(~correct_lm & ~correct_al))

    observed_diff, null, p_one, p_two = paired_sign_flip_test(
        correct_lm,
        correct_al,
        n_permutations=100_000,
        seed=RANDOM_SEED,
    )

    print()
    print("#" * 80)
    print("LM vs AL paired comparison")
    print("#" * 80)

    print(f"LM features: {lm['n_features']}")
    print(f"AL features: {al['n_features']}")
    print()
    print(f"LM accuracy:          {lm['accuracy']:.4f}")
    print(f"AL accuracy:          {al['accuracy']:.4f}")
    print(f"LM - AL accuracy:     {lm['accuracy'] - al['accuracy']:+.4f}")
    print()
    print(f"LM balanced accuracy: {lm['balanced_accuracy']:.4f}")
    print(f"AL balanced accuracy: {al['balanced_accuracy']:.4f}")
    print(f"LM - AL bal acc:      {lm['balanced_accuracy'] - al['balanced_accuracy']:+.4f}")
    print()
    print(f"LM AUC:               {lm['auc']:.4f}")
    print(f"AL AUC:               {al['auc']:.4f}")
    print(f"LM - AL AUC:          {lm['auc'] - al['auc']:+.4f}")

    print()
    print("Paired correctness table:")
    print(f"  both correct: {both_correct}")
    print(f"  LM correct, AL wrong: {lm_only}")
    print(f"  AL correct, LM wrong: {al_only}")
    print(f"  both wrong: {both_wrong}")

    print()
    print("Paired sign-flip test on per-image correctness:")
    print(f"Observed mean correctness difference, LM - AL: {observed_diff:+.6f}")
    print(f"Null mean: {null.mean():+.6f}")
    print(f"Null std:  {null.std(ddof=1):.6f}")
    print(f"One-sided p-value, LM > AL: {p_one:.8f}")
    print(f"Two-sided p-value:          {p_two:.8f}")

    if lm["accuracy"] > al["accuracy"]:
        print()
        print("Verdict goblin: LM decoded higher than AL on raw accuracy.")
    elif lm["accuracy"] < al["accuracy"]:
        print()
        print("Verdict goblin: AL decoded higher than LM on raw accuracy.")
    else:
        print()
        print("Verdict goblin: LM and AL tied on raw accuracy.")

    return {
        "correct_lm": correct_lm,
        "correct_al": correct_al,
        "lm_only": lm_only,
        "al_only": al_only,
        "both_correct": both_correct,
        "both_wrong": both_wrong,
        "observed_correctness_diff_lm_minus_al": observed_diff,
        "null_correctness_diff_lm_minus_al": null,
        "p_one_sided_lm_gt_al": p_one,
        "p_two_sided": p_two,
    }


# =============================================================================
# Main
# =============================================================================

def main():
    print()
    print("#" * 80)
    print("LM vs AL Adam logistic decoder")
    print("#" * 80)

    X, y, brain_area, labeled_mask, original_indices, good_cols = load_and_align_data()

    lm_mask = np.isin(brain_area, list(LM_ALIASES))
    al_mask = np.isin(brain_area, list(AL_ALIASES))

    print()
    print("=" * 80)
    print("Area feature counts after cleaning")
    print("=" * 80)
    print(f"LM aliases: {sorted(LM_ALIASES)}")
    print(f"AL aliases: {sorted(AL_ALIASES)}")
    print(f"LM feature count: {int(np.sum(lm_mask))}")
    print(f"AL feature count: {int(np.sum(al_mask))}")

    if np.sum(lm_mask) == 0:
        raise ValueError(
            "No LM/VISl features found. Printout above shows the available area names. "
            "Edit LM_ALIASES to match your brain_area.npy strings."
        )

    if np.sum(al_mask) == 0:
        raise ValueError(
            "No AL/VISal features found. Printout above shows the available area names. "
            "Edit AL_ALIASES to match your brain_area.npy strings."
        )

    X_lm = X[:, lm_mask]
    X_al = X[:, al_mask]

    lm = run_loo_decoder_for_area(X_lm, y, area_name="LM_or_VISl")
    al = run_loo_decoder_for_area(X_al, y, area_name="AL_or_VISal")

    comparison = summarize_comparison(y, lm, al)

    np.savez_compressed(
        OUT_NPZ,
        y=y,
        labeled_mask=labeled_mask,
        original_indices=original_indices,
        good_cols=good_cols,
        brain_area_clean=brain_area,
        lm_mask_clean=lm_mask,
        al_mask_clean=al_mask,
        lm_aliases=np.array(sorted(LM_ALIASES), dtype=object),
        al_aliases=np.array(sorted(AL_ALIASES), dtype=object),

        lm_n_features=lm["n_features"],
        lm_logits=lm["logits"],
        lm_probs=lm["probs"],
        lm_preds=lm["preds"],
        lm_accuracy=lm["accuracy"],
        lm_balanced_accuracy=lm["balanced_accuracy"],
        lm_auc=lm["auc"],
        lm_confusion_matrix=lm["confusion_matrix"],

        al_n_features=al["n_features"],
        al_logits=al["logits"],
        al_probs=al["probs"],
        al_preds=al["preds"],
        al_accuracy=al["accuracy"],
        al_balanced_accuracy=al["balanced_accuracy"],
        al_auc=al["auc"],
        al_confusion_matrix=al["confusion_matrix"],

        correct_lm=comparison["correct_lm"],
        correct_al=comparison["correct_al"],
        lm_only=comparison["lm_only"],
        al_only=comparison["al_only"],
        both_correct=comparison["both_correct"],
        both_wrong=comparison["both_wrong"],
        observed_correctness_diff_lm_minus_al=comparison["observed_correctness_diff_lm_minus_al"],
        null_correctness_diff_lm_minus_al=comparison["null_correctness_diff_lm_minus_al"],
        p_one_sided_lm_gt_al=comparison["p_one_sided_lm_gt_al"],
        p_two_sided=comparison["p_two_sided"],

        lr=LR,
        weight_decay=WEIGHT_DECAY,
        epochs=EPOCHS,
        random_seed=RANDOM_SEED,
    )

    print()
    print("=" * 80)
    print("Saved results")
    print("=" * 80)
    print(OUT_NPZ)

    print()
    print("Done.")


if __name__ == "__main__":
    main()


################################################################################
LM vs AL Adam logistic decoder
################################################################################
Loaded labels: (118,)
Raw label counts including possible -1:
{0: 62, 1: 56}
Convention assumed: 0 = inanimate, 1 = animate, -1 = unlabeled/exclude
Loaded brain areas: (39209,)
Brain area counts:
  VISal: 4249
  VISam: 2040
  VISl: 8323
  VISp: 14382
  VISpm: 4771
  VISrl: 5444

Raw data shapes
X shape:          (39209, 118)
labels shape:     (118,)
brain_area shape: (39209,)
[INFO] Transposing X to images x features.

After excluding label -1
X labeled shape: (118, 39209)
y labeled shape: (118,)
Label counts [inanimate, animate]: [62 56]

After feature cleaning
X clean shape: (118, 39209)
Removed bad/nonconstant features: 0

Area feature counts after cleaning
LM aliases: ['LM', 'VIS-l', 'VISL', 'VIS_l', 'VISl', 'visl']
AL aliases: ['AL', 'VIS-al', 'VISAL', 'VIS_al', 'VISal', 'visal']
LM feature